In [0]:
# Environment configuration - reads from job parameter or defaults to dev
import os

# Get environment from widget (job parameter) or default to 'dev'
dbutils.widgets.text("environment", "dev", "Environment (dev/prod)")
environment = dbutils.widgets.get("environment")

print(f"Running in {environment.upper()} environment")
print(f"Table: {environment}.stock_daily_data")

In [0]:
# Remove duplicates from the table based on Stock and fetch_date
# Keep the most recent record for each combination
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

# Read current data
df_current = spark.table(f"{environment}.stock_daily_data")

# Create window to identify duplicates
window_spec = Window.partitionBy("Stock", "fetch_date").orderBy(col("Stock"))

# Add row number and keep only the first occurrence
df_deduped = df_current.withColumn("row_num", row_number().over(window_spec)) \
                       .filter(col("row_num") == 1) \
                       .drop("row_num")

# Overwrite the table with deduplicated data
df_deduped.write.format("delta") \
          .mode("overwrite") \
          .saveAsTable(f"{environment}.stock_daily_data")

print(f"Deduplicated table. Original rows: {df_current.count()}, After dedup: {df_deduped.count()}")

Deduplicated table. Original rows: 3, After dedup: 3


In [0]:
from pyspark.sql.functions import round, format_number, col

df = spark.table(f"{environment}.stock_daily_data")
df_transformed = df.withColumn("Open", round(col("Open"), 2)) \
                   .withColumn("High", round(col("High"), 2)) \
                   .withColumn("Low", round(col("Low"), 2)) \
                   .withColumn("Close", round(col("Close"), 2)) \
                   .withColumn("Volume", format_number(col("Volume") / 1_000_000, 2))

display(df_transformed)

Stock,Open,High,Low,Close,Volume,fetch_date
INFY.NS,1119.0,1146.1,1118.0,1142.5,12.10,2026-05-18
RELIANCE.NS,1334.0,1342.0,1318.7,1335.9,13.02,2026-05-18
TCS.NS,2264.0,2292.9,2252.2,2283.2,3.77,2026-05-18


In [0]:
# Read the deduplicated data
df = spark.table(f"{environment}.stock_daily_data")

# Apply transformations - round price columns to 2 decimals
# Keep Volume as bigint (no formatting here)
df_transformed = df.withColumn("Open", round(col("Open"), 2)) \
                   .withColumn("High", round(col("High"), 2)) \
                   .withColumn("Low", round(col("Low"), 2)) \
                   .withColumn("Close", round(col("Close"), 2))

# Overwrite the table with transformed data
df_transformed.write.format("delta") \
              .mode("overwrite") \
              .saveAsTable(f"{environment}.stock_daily_data")

print(f"Successfully updated table with {df_transformed.count()} transformed records")

# Display the updated data
display(df_transformed)

Successfully updated table with 3 transformed records


Stock,Open,High,Low,Close,Volume,fetch_date
INFY.NS,1119.0,1146.1,1118.0,1142.5,12095591,2026-05-18
RELIANCE.NS,1334.0,1342.0,1318.7,1335.9,13021537,2026-05-18
TCS.NS,2264.0,2292.9,2252.2,2283.2,3772966,2026-05-18


In [0]:
%sql
select * from ${environment}.stock_daily_data

Stock,Open,High,Low,Close,Volume,fetch_date
INFY.NS,1119.0,1146.1,1118.0,1142.5,12095591,2026-05-18
RELIANCE.NS,1334.0,1342.0,1318.7,1335.9,13021537,2026-05-18
TCS.NS,2264.0,2292.9,2252.2,2283.2,3772966,2026-05-18
